In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline

# 1) Load dataset
df = pd.read_csv("Capstone Dataset (New).csv")

# 2) Separate features (X) and target (y)
X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

# 3) Leave-One-Out Method
loo = LeaveOneOut()

y_true_all = []
y_pred_all = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # KNN Regressor with scaling 
    model = make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(
            n_neighbors=5,     
            weights="distance", 
            p=2,                
            n_jobs=-1
        )
    )

    # Fit model
    model.fit(X_train, y_train)

    # Predict the held-out sample
    y_pred = model.predict(X_test)[0]

    # Collect predictions and true values
    y_pred_all.append(y_pred)
    y_true_all.append(y_test.values[0])

# 4) Evaluation (overall MAE across all LOO folds)
mae = mean_absolute_error(y_true_all, y_pred_all)
print("MAE:", mae)

MAE: 35.99767093979693


In [2]:
import pandas as pd

from sklearn.model_selection import GridSearchCV, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# Load data  #GridSearch #KNN #LOO #letswork
df = pd.read_csv("Capstone Dataset (New).csv")

# Split features/target  #cleanup
X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

# Pipeline: scale inside CV, then KNN  #noleakage #distancebased
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_jobs=-1))
])

# Search space  #knobs
param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],  # Manhattan vs Euclidean  #geometry
}

# Leave-One-Out CV  #oneoutatatime #tightsplit
cv = LeaveOneOut()

# Grid search  #turnthecrank
gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

# Fit search  #receipts
gs.fit(X, y)

print("Best params:", gs.best_params_)             # voila!
print("CV best MAE:", -gs.best_score_)             

Fitting 86 folds for each of 24 candidates, totalling 2064 fits
Best params: {'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'distance'}
CV best MAE: 35.99767093979693
